In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [4]:
!pip install mlflow dagshub -q

import mlflow
import dagshub
from kaggle_secrets import UserSecretsClient
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import mlflow.xgboost

user_secrets = UserSecretsClient()
dagshub_token = user_secrets.get_secret("ML_assn1")   # your DagsHub token
dagshub_username = user_secrets.get_secret("username")  # your DagsHub username

os.environ["MLFLOW_TRACKING_USERNAME"] = dagshub_username
os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token

repo_owner = dagshub_username
repo_name = "ml_assn2"

mlflow.set_tracking_uri(f"https://dagshub.com/{repo_owner}/{repo_name}.mlflow")
mlflow.set_registry_uri(f"https://dagshub.com/{repo_owner}/{repo_name}.mlflow")

dagshub.init(repo_name=repo_name, repo_owner=repo_owner, mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=67259a0a-f2b0-425c-8a2c-fb90a06628b4&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=dd06c26f4c8b1bfcf572d5512cf4927c506fe46b6530ab08dfd55c440d64e125




Accessing as lkuch23

Initialized MLflow to track repo "lkuch23/ml_assn2"

Repository lkuch23/ml_assn2 initialized!

In [5]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')
train = train_transaction.merge(train_identity, on='TransactionID', how='left')
test = test_transaction.merge(test_identity, on='TransactionID', how='left')

# Cleaning

In [6]:
missing_ratio = train.isnull().mean()
cols_to_drop = missing_ratio[missing_ratio > 0.85].index
train.drop(cols_to_drop, axis=1, inplace=True)
test.drop(cols_to_drop, axis=1, inplace=True, errors='ignore')

In [7]:
train.fillna(-999, inplace=True)
test.fillna(-999, inplace=True)

# Feature Engineering

In [8]:
train['hour'] = (train['TransactionDT'] // 3600) % 24
test['hour'] = (test['TransactionDT'] // 3600) % 24
train['day_of_week'] = (train['TransactionDT'] // (3600*24)) % 7
test['day_of_week'] = (test['TransactionDT'] // (3600*24)) % 7
train['amt_log'] = np.log1p(train['TransactionAmt'])
test['amt_log'] = np.log1p(test['TransactionAmt'])
train['amt_cents'] = train['TransactionAmt'] - train['TransactionAmt'].astype(int)
test['amt_cents'] = test['TransactionAmt'] - test['TransactionAmt'].astype(int)
train['amt_per_hour'] = train['TransactionAmt'] / (train['hour'] + 1)
test['amt_per_hour'] = test['TransactionAmt'] / (test['hour'] + 1)

In [9]:
for col in ['card1', 'card2', 'card3', 'card5']:
    if col in train.columns:
        grp = train.groupby(col)['TransactionAmt'].agg(['mean', 'std', 'count'])
        grp.columns = [f'{col}_amt_mean', f'{col}_amt_std', f'{col}_count']
        train = train.join(grp, on=col)
        test  = test.join(grp, on=col)

train.fillna(-999, inplace=True)
test.fillna(-999, inplace=True)

In [10]:
cat_cols = train.select_dtypes(include='object').columns
for col in cat_cols:
    if col in test.columns:
        le = LabelEncoder()
        combined = pd.concat([train[col], test[col]]).astype(str)
        le.fit(combined)
        train[col] = le.transform(train[col].astype(str))
        test[col] = le.transform(test[col].astype(str))
    else:
        train[col] = train[col].astype(str)
        train[col] = train[col].factorize()[0]

# Feature Selection

In [11]:
target = 'isFraud'
ID = 'TransactionID'
common_cols = set(train.columns) & set(test.columns)
features = [c for c in common_cols if c not in [target, ID]]
features = sorted(features)
X = train[features].apply(pd.to_numeric, errors='coerce').fillna(-999)
y = train[target]
X_test = test[features].apply(pd.to_numeric, errors='coerce').fillna(-999)

# Training

In [12]:
params = {
    'n_estimators': 1000,
    'max_depth': 8,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'scale_pos_weight': int((y == 0).sum() / (y == 1).sum()),
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'random_state': 42,
    'n_jobs': -1,
}

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_aucs = []

mlflow.set_experiment("IEEE_XGBoost")

with mlflow.start_run():
    for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y)):
        X_trn, X_val = X.iloc[trn_idx], X.iloc[val_idx]
        y_trn, y_val = y.iloc[trn_idx], y.iloc[val_idx]
        model = XGBClassifier(**params, early_stopping_rounds=50)
        model.fit(X_trn, y_trn, eval_set=[(X_val, y_val)], verbose=100)
        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        test_preds += model.predict_proba(X_test)[:, 1] / N_FOLDS
        fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
        fold_aucs.append(fold_auc)
        
    oof_auc = roc_auc_score(y, oof_preds)
    mlflow.log_metric("AUC", oof_auc)
    print(f"AUC: {oof_auc}")
    mlflow.xgboost.log_model(model, artifact_path="model", registered_model_name="IEEE_XGBoost_Best")

[0]	validation_0-auc:0.85751
[100]	validation_0-auc:0.93411
[200]	validation_0-auc:0.95212
[300]	validation_0-auc:0.95869
[400]	validation_0-auc:0.96252
[500]	validation_0-auc:0.96453
[600]	validation_0-auc:0.96613
[700]	validation_0-auc:0.96743
[800]	validation_0-auc:0.96841
[900]	validation_0-auc:0.96906
[999]	validation_0-auc:0.96954
[0]	validation_0-auc:0.85606
[100]	validation_0-auc:0.93591
[200]	validation_0-auc:0.95334
[300]	validation_0-auc:0.96013
[400]	validation_0-auc:0.96384
[500]	validation_0-auc:0.96651
[600]	validation_0-auc:0.96823
[700]	validation_0-auc:0.96920
[800]	validation_0-auc:0.97013
[900]	validation_0-auc:0.97074
[999]	validation_0-auc:0.97139
[0]	validation_0-auc:0.85270
[100]	validation_0-auc:0.93179
[200]	validation_0-auc:0.94860
[300]	validation_0-auc:0.95673
[400]	validation_0-auc:0.96110
[500]	validation_0-auc:0.96396
[600]	validation_0-auc:0.96570
[700]	validation_0-auc:0.96730
[800]	validation_0-auc:0.96837
[900]	validation_0-auc:0.96911
[999]	validati

2026/05/05 21:00:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


AUC: 0.9706934455226585


Registered model 'IEEE_XGBoost_Best' already exists. Creating a new version of this model...
2026/05/05 21:00:24 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: IEEE_XGBoost_Best, version 2
Created version '2' of model 'IEEE_XGBoost_Best'.


🏃 View run auspicious-kit-212 at: https://dagshub.com/lkuch23/ml_assn2.mlflow/#/experiments/3/runs/a6cfb619df30438d89fe6defbb61ab68
🧪 View experiment at: https://dagshub.com/lkuch23/ml_assn2.mlflow/#/experiments/3
